#  Cleaning Data Using SQL - Automobile Dataset

## Activity overview 

When it comes to data stored in databases, that means using SQL queries. In this activity, we will create a custom dataset and table, import a CSV file, and use SQL queries to clean automobile data.

In this scenario, we are a data analyst working with a used car dealership startup venture. The investors want us to find out which cars are most popular with customers so they can make sure to stock accordingly. 

## Objective: clean data using SQL. 
This will enable us to process and analyze data in databases, which is a common task for data analysts.

## Table of Contents:
- Create Table
- Clean Data
    - Step 1: Inspect the fuel_type column
    - Step 2: Inspect the length column
    - Step 3: Fill in missing data
    - Step 4: Identify potential errors
    - Step 5: Ensure consistency

## Create Table
Here we use a local csv file, so I use DuckDB for easier import. 
Let's name our table "car_info"

In [1]:
!pip install duckdb --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [2]:
import duckdb

con = duckdb.connect("cars.duckdb")
con.sql("""
CREATE TABLE car_info AS
SELECT *
FROM read_csv_auto('automobile_data.csv')
""")

In [3]:
con.sql("""
SELECT *
FROM car_info
LIMIT 10;
""")

┌─────────────┬───────────┬──────────────┬─────────────┬──────────────┬─────────────────┬────────────┬────────┬────────┬────────┬─────────────┬─────────────┬──────────────────┬─────────────┬─────────────┬───────────────────┬────────────┬──────────┬─────────────┬───────┐
│    make     │ fuel_type │ num_of_doors │ body_style  │ drive_wheels │ engine_location │ wheel_base │ length │ width  │ height │ curb_weight │ engine_type │ num_of_cylinders │ engine_size │ fuel_system │ compression_ratio │ horsepower │ city_mpg │ highway_mpg │ price │
│   varchar   │  varchar  │   varchar    │   varchar   │   varchar    │     varchar     │   double   │ double │ double │ double │    int64    │   varchar   │     varchar      │    int64    │   varchar   │      double       │   int64    │  int64   │    int64    │ int64 │
├─────────────┼───────────┼──────────────┼─────────────┼──────────────┼─────────────────┼────────────┼────────┼────────┼────────┼─────────────┼─────────────┼──────────────────┼───────────

## Clean Data
Our new dataset contains historical sales data, including details such as car features and prices. We can use this data to find the top 10 most popular cars and trims. But before we can perform our analysis, we’ll need to make sure our data is clean. If we analyze dirty data, we could end up presenting the wrong list of cars to the investors. That may cause them to lose money on their car inventory investment.

### Step 1: Inspect the fuel_type column
The first thing we want to do is inspect the data in our table so we can find out if there is any specific cleaning that needs to be done. \
According to the data’s description, the fuel_type column should only have two unique string values: diesel and gas. 

In [4]:
con.sql("""
SELECT
  DISTINCT fuel_type
FROM
  car_info;
""")

┌───────────┐
│ fuel_type │
│  varchar  │
├───────────┤
│ diesel    │
│ gas       │
└───────────┘

The result confirms that the fuel_type column doesn’t have any unexpected values. 

### Step 2: Inspect the length column 
Next, we will inspect a column with numerical data. \
The length column should contain numeric measurements of the cars. \
So we will check that the minimum and maximum lengths in the dataset align with the data description, which states that the lengths in this column should range from 141.1 to 208.1. 

In [5]:
con.sql("""
SELECT
  MIN(length) AS min_length,
  MAX(length) AS max_length
FROM
  car_info;
""")

┌────────────┬────────────┐
│ min_length │ max_length │
│   double   │   double   │
├────────────┼────────────┤
│      141.1 │      208.1 │
└────────────┴────────────┘

Our results confirm that 141.1 and 208.1 are the minimum and maximum values respectively in this column. 

### Step 3: Fill in missing data
Missing values can create errors or skew our results during analysis. \
We’re going to want to check our data for null or missing values. \
These values might appear as a blank cell or the word null. \
We can check to see if the num_of_doors column contains null values: 

In [6]:
#This will select any rows with missing data for the num_of_doors column and return them in our results table.
con.sql("""
SELECT
  *
FROM
  car_info 
WHERE 
  num_of_doors IS NULL;
""")

┌─────────┬───────────┬──────────────┬────────────┬──────────────┬─────────────────┬────────────┬────────┬────────┬────────┬─────────────┬─────────────┬──────────────────┬─────────────┬─────────────┬───────────────────┬────────────┬──────────┬─────────────┬───────┐
│  make   │ fuel_type │ num_of_doors │ body_style │ drive_wheels │ engine_location │ wheel_base │ length │ width  │ height │ curb_weight │ engine_type │ num_of_cylinders │ engine_size │ fuel_system │ compression_ratio │ horsepower │ city_mpg │ highway_mpg │ price │
│ varchar │  varchar  │   varchar    │  varchar   │   varchar    │     varchar     │   double   │ double │ double │ double │    int64    │   varchar   │     varchar      │    int64    │   varchar   │      double       │   int64    │  int64   │    int64    │ int64 │
├─────────┼───────────┼──────────────┼────────────┼──────────────┼─────────────────┼────────────┼────────┼────────┼────────┼─────────────┼─────────────┼──────────────────┼─────────────┼─────────────┼───

 We got two results, one Mazda and one Dodge

In order to fill in these missing values, we check with the sales manager, who states that all Dodge gas sedans and all Mazda diesel sedans sold had four doors. \
Let's update our table so that all Dodge gas sedans have four doors:

In [10]:
con.sql("""
UPDATE
  car_info
SET
  num_of_doors = 'four'
WHERE
  make = 'dodge'
  AND fuel_type = 'gas'
  AND body_style = 'sedan';
""")

Three rows were modified in this table. 

To make sure, we run the previous query again:

In [12]:
con.sql("""
SELECT
  *
FROM
  car_info 
WHERE 
  num_of_doors IS NULL;
""")

┌─────────┬───────────┬──────────────┬────────────┬──────────────┬─────────────────┬────────────┬────────┬────────┬────────┬─────────────┬─────────────┬──────────────────┬─────────────┬─────────────┬───────────────────┬────────────┬──────────┬─────────────┬───────┐
│  make   │ fuel_type │ num_of_doors │ body_style │ drive_wheels │ engine_location │ wheel_base │ length │ width  │ height │ curb_weight │ engine_type │ num_of_cylinders │ engine_size │ fuel_system │ compression_ratio │ horsepower │ city_mpg │ highway_mpg │ price │
│ varchar │  varchar  │   varchar    │  varchar   │   varchar    │     varchar     │   double   │ double │ double │ double │    int64    │   varchar   │     varchar      │    int64    │   varchar   │      double       │   int64    │  int64   │    int64    │ int64 │
├─────────┼───────────┼──────────────┼────────────┼──────────────┼─────────────────┼────────────┼────────┼────────┼────────┼─────────────┼─────────────┼──────────────────┼─────────────┼─────────────┼───

Now, we only have one row with a NULL value for num_of_doors. 

Let's Repeat this process to replace the null value for the Mazda. 

In [13]:
con.sql("""
UPDATE
  cars.car_info
SET
  num_of_doors = 'four'
WHERE
  make = 'mazda'
  AND fuel_type = 'diesel'
  AND body_style = 'sedan';
""")

In [14]:
con.sql("""
SELECT
  *
FROM
  car_info 
WHERE 
  num_of_doors IS NULL;
""")

┌─────────┬───────────┬──────────────┬────────────┬──────────────┬─────────────────┬────────────┬────────┬────────┬────────┬─────────────┬─────────────┬──────────────────┬─────────────┬─────────────┬───────────────────┬────────────┬──────────┬─────────────┬───────┐
│  make   │ fuel_type │ num_of_doors │ body_style │ drive_wheels │ engine_location │ wheel_base │ length │ width  │ height │ curb_weight │ engine_type │ num_of_cylinders │ engine_size │ fuel_system │ compression_ratio │ horsepower │ city_mpg │ highway_mpg │ price │
│ varchar │  varchar  │   varchar    │  varchar   │   varchar    │     varchar     │   double   │ double │ double │ double │    int64    │   varchar   │     varchar      │    int64    │   varchar   │      double       │   int64    │  int64   │    int64    │ int64 │
└─────────┴───────────┴──────────────┴────────────┴──────────────┴─────────────────┴────────────┴────────┴────────┴────────┴─────────────┴─────────────┴──────────────────┴─────────────┴─────────────┴───

### Step 4: Identify potential errors
Once we have finished ensuring that there aren’t any missing values in our data, we’ll want to check for other potential errors. 
We can use SELECT DISTINCT to check what values exist in a column. 

Let's check the num_of_cylinders column: 

In [15]:
con.sql("""
SELECT
  DISTINCT num_of_cylinders
FROM
  car_info;
""")

┌──────────────────┐
│ num_of_cylinders │
│     varchar      │
├──────────────────┤
│ five             │
│ two              │
│ three            │
│ four             │
│ six              │
│ tow              │
│ eight            │
│ twelve           │
└──────────────────┘

After running this, we notice that there are one too many rows. 

There are two entries for two cylinders: rows 3 and 7. But the two in row 7 is misspelled. \
To correct the misspelling for all rows, we can update the column: 

In [17]:
con.sql("""
UPDATE
  car_info
SET
  num_of_cylinders = 'two'
WHERE
  num_of_cylinders = 'tow';
""")

One row was modified after running this statement. 

To check that it worked, we can run the previous query again: 

In [19]:
con.sql("""
SELECT
  DISTINCT num_of_cylinders
FROM
  car_info;
""")

┌──────────────────┐
│ num_of_cylinders │
│     varchar      │
├──────────────────┤
│ twelve           │
│ eight            │
│ three            │
│ five             │
│ two              │
│ four             │
│ six              │
└──────────────────┘

Next, we can check the compression_ratio column. \
According to the data description, the compression_ratio column values should range from 7 to 23. \
Just like when we checked the length values , we can use MIN and MAX to check if that’s correct: 

In [20]:
con.sql("""
SELECT
  MIN(compression_ratio) AS min_compression_ratio,
  MAX(compression_ratio) AS max_compression_ratio
FROM
  car_info;
""")

┌───────────────────────┬───────────────────────┐
│ min_compression_ratio │ max_compression_ratio │
│        double         │        double         │
├───────────────────────┼───────────────────────┤
│                   7.0 │                  70.0 │
└───────────────────────┴───────────────────────┘

Notice that this returns a maximum of 70. \
But we know this is an error because the maximum value in this column should be 23, not 70. So the 70 is most likely a 7.0. 

In [21]:
#Run the above query again without the row with 70 to make sure that the rest of the values fall within the expected range of 7 to 23.
con.sql("""
SELECT
  MIN(compression_ratio) AS min_compression_ratio,
  MAX(compression_ratio) AS max_compression_ratio
FROM
  car_info
WHERE
  compression_ratio <> 70;
""")

┌───────────────────────┬───────────────────────┐
│ min_compression_ratio │ max_compression_ratio │
│        double         │        double         │
├───────────────────────┼───────────────────────┤
│                   7.0 │                  23.0 │
└───────────────────────┴───────────────────────┘

Now the highest value is 23, which aligns with the data description. So we’ll want to correct the 70 value. \
Suppose that we checked with the sales manager again, who said that this row was made in error and should be removed. 

Before we delete anything, we should check to see how many rows contain this erroneous value as a precaution so that we don’t end up deleting 50% of our data. \
If there are too many (for instance, 20% of our rows have the incorrect 70 value), then we would want to check back in with the sales manager to inquire if these should be deleted or if the 70 should be updated to another value. 

Let's count how many rows we would be deleting:

In [22]:
con.sql("""
SELECT
   COUNT(*) AS num_of_rows_to_delete
FROM
   car_info
WHERE
   compression_ratio = 70;
""")

┌───────────────────────┐
│ num_of_rows_to_delete │
│         int64         │
├───────────────────────┤
│                     1 │
└───────────────────────┘

Turns out there is only one row with the erroneous 70 value. 

So we can delete that row:  

In [24]:
con.sql("""
DELETE FROM car_info
WHERE compression_ratio = 70;
""")

In [25]:
#To check that it worked, run the previous query again:
con.sql("""
SELECT
   COUNT(*) AS num_of_rows_to_delete
FROM
   car_info
WHERE
   compression_ratio = 70;
""")

┌───────────────────────┐
│ num_of_rows_to_delete │
│         int64         │
├───────────────────────┤
│                     0 │
└───────────────────────┘

### Step 5: Ensure consistency

Finally, we want to check our data for any inconsistencies that might cause errors. \
These inconsistencies can be tricky to spot — sometimes even something as simple as an extra space can cause a problem.

Check the drive_wheels column for inconsistencies by running a query with a SELECT DISTINCT statement: 

In [26]:
con.sql("""
SELECT
  DISTINCT drive_wheels
FROM
  car_info;
""")

┌──────────────┐
│ drive_wheels │
│   varchar    │
├──────────────┤
│ fwd          │
│ 4wd          │
│ rwd          │
│ 4wd          │
└──────────────┘

It appears that 4wd appears twice in results. \
However, because we used a SELECT DISTINCT statement to return unique values, this probably means there’s an extra space in one of the 4wd entries that makes it different from the other 4wd. 

To check if this is the case, we can use a LENGTH statement to determine the length of how long each of these string variables: 

In [27]:
con.sql("""
SELECT
  DISTINCT drive_wheels,
  LENGTH(drive_wheels) AS string_length
FROM
  car_info;
""")

┌──────────────┬───────────────┐
│ drive_wheels │ string_length │
│   varchar    │     int64     │
├──────────────┼───────────────┤
│ fwd          │             3 │
│ 4wd          │             4 │
│ rwd          │             3 │
│ 4wd          │             3 │
└──────────────┴───────────────┘

According to these results, some instances of the 4wd string have four characters instead of the expected three (4wd has 3 characters). 

In that case, we can use the TRIM function to remove all extra spaces in the drive_wheels column: 

In [28]:
con.sql("""
UPDATE
  car_info
SET
  drive_wheels = TRIM(drive_wheels)
WHERE TRUE;
""")

Then, we run the SELECT DISTINCT statement again to ensure that there are only three distinct values in the drive_wheels column: 

In [29]:
con.sql("""
SELECT
  DISTINCT drive_wheels,
  LENGTH(drive_wheels) AS string_length
FROM
  car_info;
""")

┌──────────────┬───────────────┐
│ drive_wheels │ string_length │
│   varchar    │     int64     │
├──────────────┼───────────────┤
│ rwd          │             3 │
│ 4wd          │             3 │
│ fwd          │             3 │
└──────────────┴───────────────┘

And now there should only be three unique values in this column! Which means our data is clean, consistent, and ready for analysis! 